## Structured Output

models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily
parsed and used in subsequent processing.

Langchain uspports muliple schemas types and methods for enforcing structured output.

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [ ]:
pip install -U langchain-google-genai langchain -quiet

Note: you may need to restart the kernel to use updated packages.


c:\Users\chsha\Desktop\KrishNaik Course\Lanchain_Updated\.venv\Scripts\python.exe: No module named pip


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Set your API key (get this from Google AI Studio)
os.environ["GEMINI_API_KEY"] = 

# Initialize the model (using gemini-3.5-flash, the engine behind Antigravity)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.7,
    max_retries=2
)

# Invoke the model
# message = HumanMessage(content="Write a simple Python script to reverse a string.")
# response = llm.invoke([message])

# print(response.content)

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This is the year movie was released")
    director:str=Field(description="This is the director of the movie")
    ratings:float=Field(description="Movies ratings out of 10")

model_with_Structured = llm.with_structured_output(Movie)

response = model_with_Structured.invoke("provide details of moview baahubali")
print(response)

title='Baahubali: The Beginning' year=2015 director='S. S. Rajamouli' ratings=8.0


In [9]:
## Message output alongside parsed structure
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(..., description="The title of the movie")
    year:int=Field(..., description="This is the year movie was released")
    director:str=Field(..., description="This is the director of the movie")
    ratings:float=Field(..., description="Movies ratings out of 10")

model_with_Structured = llm.with_structured_output(Movie, include_raw=True)

response = model_with_Structured.invoke("provide details of moview baahubali")
print(response)

{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "title": "Baahubali: The Beginning",\n  "year": 2015,\n  "director": "S. S. Rajamouli",\n  "ratings": 8.0\n}', 'extras': {'signature': 'EjQKMgEMOdbH9htYEJBz4DQQNouxgDmpjrW00PRAH7kNc15eIy/iGzPc2P71IgT1zosiMJxA'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e5109-a5a2-7172-b6c6-b5588fc7fefe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 51, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}}), 'parsed': Movie(title='Baahubali: The Beginning', year=2015, director='S. S. Rajamouli', ratings=8.0), 'parsing_error': None}


## Nested Structure

In [10]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres : list[str]
    budject: float | None = Field(None, description="Budget in million USD")

model_with_Structured = llm.with_structured_output(MovieDetails)

response = model_with_Structured.invoke("Provide Details about Baahubali movie")
response



MovieDetails(title='Baahubali: The Beginning', year=2015, cast=[Actor(name='Prabhas', role='Amarendra Baahubali / Shivudu'), Actor(name='Rana Daggubati', role='Bhallaladeva'), Actor(name='Anushka Shetty', role='Devasena'), Actor(name='Tamannaah Bhatia', role='Avanthika'), Actor(name='Ramya Krishnan', role='Sivagami')], genres=['Action', 'Adventure', 'Drama', 'Fantasy'], budject=28.0)

## TypedDict

TypedDict provides a simpler alternative using python's built-in typing, ideal when you dont need runtime validation.

In [13]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"This is the year movie was released"]
    director:Annotated[str,...,"This is the director of the movie"]
    ratings:Annotated[float,...,"Movies ratings out of 10"]


model_with_Structured = llm.with_structured_output(MovieDict)

response = model_with_Structured.invoke("Provide Details about Baahubali movie")
response  

{'title': 'Baahubali: The Beginning',
 'year': 2015,
 'director': 'S. S. Rajamouli',
 'ratings': 8.0}

In [14]:
class Actor(TypedDict):
    name : str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres : list[str]
    budject: float | None = Field(None, description="Budget in million USD")

model_with_Structured = llm.with_structured_output(MovieDetails)

response = model_with_Structured.invoke("Provide Details about Baahubali movie")
response



{'title': 'Baahubali: The Beginning',
 'year': 2015,
 'cast': [{'name': 'Prabhas',
   'role': 'Amarendra Baahubali / Mahendra Baahubali'},
  {'name': 'Rana Daggubati', 'role': 'Bhallaladeva'},
  {'name': 'Anushka Shetty', 'role': 'Devasena'},
  {'name': 'Tamannaah Bhatia', 'role': 'Avanthika'}],
 'genres': ['Action', 'Adventure', 'Drama', 'Fantasy'],
 'budject': 1800000000}

## DataClasses

A data class is a class typically containing mainly data, although there anent really any restrictions. We create it using the @dataclass decorator

In [5]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name: str = Field(description="names of the person")
    email: str = Field(description="email of the person")
    phone: str = Field(description="phone number of the person")

agent = create_agent(model=llm, response_format=ContactInfo)

results = agent.invoke(
    {"messages":[{"role":"user", "content": "Extract contact info from : John Doe, john@exmple.com, (555) 123-4567" }]}
    )

print(results["structured_response"])

name='John Doe' email='john@exmple.com' phone='(555) 123-4567'
